# URL-Feature Model Comparisons

COSC 50523 — Deep Minds  
Leiton Peterson

This notebook compares model settings using the team's prepared
URL-feature dataset and shared train/validation split.

The first experiment compares logistic regression with different
regularization strengths while keeping the data and other settings fixed.

Models are evaluated on validation data.
The test set is reserved for the team's final evaluation.

Labels:
- 0 = legitimate
- 1 = phishing

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/DeepMinds')

# The prepared dataset should be directly inside DeepMinds.
master_file = DATA_DIR / 'master_with_splits.csv'

print("Project folder found:", DATA_DIR.is_dir())
print("Master dataset found:", master_file.is_file())

# Check both locations used in the team's instructions.
helper_folders = [DATA_DIR, DATA_DIR / 'shared']

for folder in helper_folders:
    print(f"\nChecking: {folder}")

    for filename in ['project_utils.py', 'torch_models.py']:
        file_path = folder / filename
        status = 'FOUND' if file_path.is_file() else 'MISSING'
        print(f"  {filename}: {status}")

Project folder found: True
Master dataset found: True

Checking: /content/drive/MyDrive/DeepMinds
  project_utils.py: FOUND
  torch_models.py: FOUND

Checking: /content/drive/MyDrive/DeepMinds/shared
  project_utils.py: MISSING
  torch_models.py: MISSING


## Load the URL-feature data

Use the team's prepared splits and 22 URL-related features.
The shared preprocessing function fits scaling on training data only.

In [3]:
import sys
import pandas as pd

# Allow Python to import the team's helper file from Drive.
sys.path.insert(0, str(DATA_DIR))

import project_utils
from project_utils import (
    URL_FEATURES,
    load_master,
    get_feature_splits,
    evaluate_model,
)

print("Using helper file:", project_utils.__file__)

# Load the dataset with the team's existing split assignments.
df = load_master(DATA_DIR)

# Use the same feature selection and preprocessing as the other models.
data, scaler = get_feature_splits(
    df,
    features=URL_FEATURES,
    split_col='split',
)

X_train, y_train = data['train']
X_val, y_val = data['val']

print("\nNumber of features:", len(URL_FEATURES))
print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)

# Basic checks before training.
assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)
assert X_train.shape[1] == X_val.shape[1] == len(URL_FEATURES)
assert set(y_train) == {0, 1}
assert set(y_val) == {0, 1}

print("\nValidation label counts:")
print(
    pd.Series(y_val)
    .value_counts()
    .rename(index={0: 'Legitimate', 1: 'Phishing'})
)

print("\nData loaded and basic checks passed.")

Using helper file: /content/drive/MyDrive/DeepMinds/project_utils.py

Number of features: 22
Training features: (165056, 22)
Validation features: (35369, 22)

Validation label counts:
Legitimate    20227
Phishing      15142
Name: count, dtype: int64

Data loaded and basic checks passed.


## Logistic regression: regularization comparison

Compare C = 0.1, 1.0, and 10.0 while keeping the features,
split, solver, and other settings fixed.

C controls the strength of regularization:
- Smaller C means stronger regularization.
- Larger C means weaker regularization.

All results below use the validation split.

In [4]:
from sklearn.linear_model import LogisticRegression

C_values = [0.1, 1.0, 10.0]
lr_results = []

for C in C_values:
    print(f"Training logistic regression with C={C}...")

    model = LogisticRegression(
        C=C,
        solver='lbfgs',
        max_iter=1000,
        random_state=42,
    )
    model.fit(X_train, y_train)

    # Find the probability column corresponding to phishing (label 1).
    phishing_column = list(model.classes_).index(1)
    val_probabilities = model.predict_proba(X_val)[:, phishing_column]

    # Use the team's shared evaluation function.
    result = evaluate_model(
        name=f'Logistic Regression C={C}',
        split='val',
        y_true=y_val,
        y_prob=val_probabilities,
        threshold=0.5,
        show=False,
    )

    result['C'] = C
    result['Iterations'] = int(model.n_iter_[0])
    lr_results.append(result)

# Show all three experiments together.
lr_comparison = pd.DataFrame(lr_results)

columns = [
    'C', 'Split', 'Accuracy', 'Precision', 'Recall', 'F1',
    'False positives', 'False negatives', 'Iterations',
]

display(lr_comparison[columns].round(6))

Training logistic regression with C=0.1...
Training logistic regression with C=1.0...
Training logistic regression with C=10.0...


,C,Split,Accuracy,Precision,Recall,F1,False positives,False negatives,Iterations
0,0.1,val,0.999293,1.0,0.998349,0.999174,0,25,12
1,1.0,val,0.999689,1.0,0.999274,0.999637,0,11,14
2,10.0,val,0.999689,1.0,0.999274,0.999637,0,11,15


## Preliminary findings

We compared three logistic-regression C values using the same
22 features, training/validation split, and classification threshold.

C = 0.1 produced 25 false negatives. C = 1.0 and C = 10.0 each
produced 11 false negatives. All three settings had zero false positives.

Increasing C from 0.1 to 1.0 improved phishing recall. Increasing it
further to 10.0 did not improve the reported classification metrics.
C = 1.0 is therefore a reasonable setting to retain for now.

These are validation results; no test evaluation was performed.
Random Forest and MLP comparisons remain future work.